In [1]:
!pip install -q transformers langchain langchain-community chromadb sentence-transformers torch langchain-core

from transformers import pipeline
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from ipywidgets import interact, Text

# Initialize text generation model
generator = pipeline("text-generation", model="gpt2")

# Initialize embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Sample medical reports
medical_reports = [
    "Patient A: Age 45, diagnosed with hypertension. Prescribed Lisinopril 10mg daily. BP: 140/90 mmHg.",
    "Patient B: Age 62, diagnosed with diabetes type 2. On Metformin 500mg twice daily. Glucose: 180 mg/dL.",
    "Patient C: Age 33, diagnosed with asthma. Using Albuterol inhaler as needed. Symptoms stable."
]

# Sample questions
questions = [
    "What medication is Patient A taking?",
    "What is the glucose level of Patient B?",
    "What condition does Patient C have?"
]

# Convert reports to documents and build vector store
docs = [Document(page_content=report) for report in medical_reports]
vector_store = Chroma.from_documents(docs, embedding_model)

def rag_answer(question, max_tokens=100, temperature=0.7, top_p=0.9):
    retrieved_docs = vector_store.similarity_search(question, k=1)

    if retrieved_docs:
        context = retrieved_docs[0].page_content
    else:
        context = "No relevant information found."

    prompt = f"Based on this context: '{context}', answer the question: {question}"

    output = generator(
        prompt,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        truncation=True,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = output[0]["generated_text"].replace(prompt, "").strip()
    return answer

@interact
def interactive_rag(query=Text(value="What medication is Patient A taking?")):
    answer = rag_answer(query)
    print("\n" + "="*50)
    print(f"  Question: {query}")
    print("="*50)
    print(f"  Answer  : {answer}")
    print("="*50)

# Example test cases
print("\nSample Queries and Answers:\n")
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_answer(q)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...


Sample Queries and Answers:


Q: What medication is Patient A taking?


Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Patient B: Age 45, diagnosed with hypertension. Prescribed Lisinopril 10mg daily. BP: 140/90 mmHg.', answer the question: What medication is Patient B taking?

Patient C: Age 45, diagnosed with hypertension. Prescribed Lisinopril 10mg daily. BP: 140/90 mmHg.', answer the question: What medication is Patient C taking?

Patient D: Age 45, diagnosed with hypertension

Q: What is the glucose level of Patient B?


Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 'Patient B: Glucose level is measured by the ratio of fasting glucose to total cholesterol. The ratio is the sum of the two measures. A fasting glucose of 0.8 mmol/L or less is considered a normal daily glucose level. In this case, a fasting glucose of 0.6 mmol/L is considered to be normal daily glucose. In the case of a normal daily glucose, a fasting glucose of 0.8 mmol/L is considered to be normal daily glucose.

Q: What condition does Patient C have?
A: The results of this study suggest that the use of Albuterol in children with asthma is associated with an increased risk of asthma. The authors note that the study is based on a population-based sample of children aged 3 to 14 years, and that the results are based on data from the US Centers for Disease Control and Prevention (CDC).

The researchers examined whether the use of Albuterol in children with asthma was associated with a risk of developing asthma in the study population
